In [ ]:
# Install and Configure
!pip install plotly --quiet
%matplotlib inline  

#Import Libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


In [ ]:
# Model tools
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Logistic regression model
from sklearn.linear_model import LogisticRegression

In [ ]:
# Model evaluation
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve
)

# Load Data
df = pd.read_csv("loan.csv")
display(df.head())  # display() renders richer tables in Jupyter
display(df.describe())
df.info()

In [ ]:
# Exploratory Data Analysis (EDA)
# Loan status distribution
sns.countplot(x='loan_status', data=df)
plt.title("Loan Status Distribution")
plt.show()

In [ ]:
# Loan amount distribution
plt.figure(figsize=(8, 5))
sns.histplot(df['loan_amnt'], bins=30)
plt.title("Loan Amount Distribution")
plt.show()

In [ ]:
# Debt-to-income vs loan status
sns.boxplot(x='loan_status', y='dti', data=df)
plt.title("Debt to Income vs Loan Status")
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), cmap='coolwarm')  # FIX 2: numeric_only=True
plt.title("Feature Correlation")
plt.show()

In [ ]:
# Data Cleaning and Preprocessing
display(df.isnull().sum())
df = df.dropna()

In [ ]:
# Convert loan status to numeric values
df['loan_status'] = df['loan_status'].map({
    'Fully Paid': 0,
    'Charged Off': 1
})

# Clean emp_length — strip text and convert to numeric
df['emp_length'] = (
    df['emp_length']
    .str.replace(r'\+ years| years| year|< ', '', regex=True)
    .str.strip()
    .astype(float)
)

features = ['loan_amnt', 'annual_inc', 'dti', 'emp_length']
X = df[features]
y = df['loan_status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
# Scale features for better convergence
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Train Model and Predict
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

In [ ]:
# Model Evaluation
conf_matrix = confusion_matrix(y_test, y_pred)
sns.heatmap(conf_matrix, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.show()

print(classification_report(y_test, y_pred))

roc_auc = roc_auc_score(y_test, y_prob)
print("ROC-AUC Score:", roc_auc)

fpr, tpr, thresholds = roc_curve(y_test, y_prob)
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.plot([0, 1], [0, 1], '--', color='grey')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

In [ ]:
# Business Impact Evaluation
default_rate = y_pred.mean() * 100 
print(f"Predicted Default Rate: {default_rate:.2f}%")